# Policy Event Study: Funding Reform Effects on District Outcomes

**Research question:** What happens to district outcomes in the years around a state-level funding reform?

An event study plots outcomes in relative time — years before and after the policy event — to:
1. Test **pre-trends** (are treatment districts on the same trajectory before the event?)
2. Estimate **post-event effects** (do outcomes improve after the reform?)

This notebook uses the project's `event_study` module to build the relative-time panel, then visualises the results.


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Add project source to path so pipeline/reports can be imported
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PANEL_PATH = REPO / "data" / "processed" / "district_year_panel.csv"


In [ ]:
def make_synthetic_panel(n_districts: int = 300, seed: int = 42) -> pd.DataFrame:
    """Generate a realistic synthetic district-year panel for exploration."""
    rng = np.random.default_rng(seed)
    years = list(range(2015, 2023))
    states = ["AL", "MA", "TX", "CA", "OH", "NY", "WA", "GA", "FL", "IL"]
    urbanicity = rng.choice(["City", "Suburb", "Town", "Rural"], n_districts, p=[0.20, 0.30, 0.25, 0.25])
    state = [states[i % len(states)] for i in range(n_districts)]

    poverty0    = rng.beta(2, 7, n_districts)
    income0     = np.clip(90_000 - 220_000 * poverty0 + rng.normal(0, 8_000, n_districts), 28_000, 130_000)
    ba0         = np.clip(0.38 - 0.60 * poverty0 + rng.normal(0, 0.05, n_districts), 0.08, 0.75)
    spending0   = np.clip(7_000 + (income0 / 100_000) * 7_000 + rng.normal(0, 2_500, n_districts), 5_000, 32_000)
    instr_share = rng.uniform(0.52, 0.70, n_districts)
    admin_share = rng.uniform(0.05, 0.13, n_districts)
    overperform = rng.normal(0, 0.07, n_districts)   # persistent district "true quality" residual

    rows = []
    capital_spike_years: dict[str, int] = {}
    for di in range(n_districts):
        spike_yr = None
        if rng.random() < 0.18:           # ~18% of districts get a capital investment spike
            spike_yr = years[int(rng.integers(2, len(years) - 3))]
            capital_spike_years[f"D{di:04d}"] = spike_yr

        for yi, yr in enumerate(years):
            poverty  = float(np.clip(poverty0[di] + rng.normal(0, 0.008), 0.02, 0.55))
            income   = float(np.clip(income0[di] + 800 * yi + rng.normal(0, 800), 25_000, 140_000))
            ba       = float(np.clip(ba0[di] + 0.002 * yi + rng.normal(0, 0.004), 0.08, 0.75))
            spending = float(np.clip(spending0[di] + 250 * yi + rng.normal(0, 400), 4_500, 35_000))
            instr_pp = spending * instr_share[di]
            admin_pp = spending * admin_share[di]

            capital_pp = spending * float(rng.beta(1, 10)) * 0.12
            if spike_yr and yr == spike_yr:
                capital_pp = spending * float(rng.uniform(0.18, 0.35))

            # Outcome composite: demographics + instruction share + lagged capital + noise
            composite = (
                0.72
                - 1.10 * poverty
                + 0.25 * ba
                + 0.04 * (income / 80_000 - 1.0)
                + overperform[di]
                + 0.12 * (instr_share[di] - 0.61)
                + (0.045 if (spike_yr and yr >= spike_yr + 3) else 0.0)
                + 0.001 * yi
                + float(rng.normal(0, 0.025))
            )
            composite = float(np.clip(composite, 0.12, 0.97))

            rows.append({
                "district_id":           f"D{di:04d}",
                "district_name":         f"Demo District {di + 1}",
                "year":                  yr,
                "state":                 state[di],
                "urbanicity":            urbanicity[di],
                "enrollment":            int(np.clip(rng.lognormal(7.8, 1.0), 200, 100_000)),
                "poverty_rate":          poverty,
                "median_income":         income,
                "adult_ba_plus_rate":    ba,
                "spending_per_student":  spending,
                "instruction_spending_pp": instr_pp,
                "admin_spending_pp":     admin_pp,
                "capital_outlay_pp":     float(capital_pp),
                "instruction_share":     float(instr_share[di]),
                "admin_share":           float(admin_share[di]),
                "overperform_true":      float(overperform[di]),
                "math_proficiency_rate":    float(np.clip(composite * 0.92 + rng.normal(0, 0.02), 0.10, 0.98)),
                "reading_proficiency_rate": float(np.clip(composite * 0.94 + rng.normal(0, 0.02), 0.10, 0.98)),
                "graduation_rate":          float(np.clip(0.65 + composite * 0.35 + rng.normal(0, 0.015), 0.45, 0.99)),
                "attendance_rate":          float(np.clip(0.82 + composite * 0.18 + rng.normal(0, 0.010), 0.70, 0.99)),
                "capital_spike_year":    spike_yr,
            })

    df = pd.DataFrame(rows)
    df._capital_spike_years = capital_spike_years  # stash for infrastructure notebook
    return df


def load_panel() -> tuple[pd.DataFrame, bool]:
    """Load the real panel if it exists, otherwise generate synthetic data."""
    if PANEL_PATH.exists():
        df = pd.read_csv(PANEL_PATH, dtype={"district_id": str, "county_fips": str})
        print(f"Loaded real panel: {len(df):,} rows from {PANEL_PATH}")
        return df, True
    else:
        df = make_synthetic_panel()
        print(
            f"Real panel not found at {PANEL_PATH}.\n"
            f"Using synthetic data ({len(df):,} rows) — run eol-build-panel to use real data."
        )
        return df, False


def outcome_composite(df: pd.DataFrame) -> pd.Series:
    """Weighted mean of available outcome metrics (mirrors reports.py weights)."""
    weights = {
        "math_proficiency_rate":    1.0,
        "reading_proficiency_rate": 1.0,
        "graduation_rate":          1.5,
        "attendance_rate":          0.5,
    }
    num = pd.Series(0.0, index=df.index)
    den = pd.Series(0.0, index=df.index)
    for col, w in weights.items():
        if col in df.columns:
            valid = pd.to_numeric(df[col], errors="coerce")
            mask  = valid.notna()
            num  += valid.where(mask, 0) * w
            den  += mask.astype(float) * w
    return (num / den.replace(0, np.nan)).rename("outcome_composite")


In [ ]:
from education_opportunity_lab.event_study import add_relative_time, demean_within_district

df, is_real = load_panel()
df["outcome_composite"] = outcome_composite(df)
df = df.sort_values(["district_id", "year"])

if is_real:
    # Try to read policy events from the real panel data directory
    events_path = REPO / "samples" / "policy_events.csv"
    if events_path.exists():
        events = pd.read_csv(events_path)
        print(f"Loaded {len(events)} policy events from {events_path}")
    else:
        events = None
        print("No policy_events.csv found — will generate synthetic events")
else:
    events = None
    print("Using synthetic panel — generating synthetic funding reform events")


In [ ]:
if events is None:
    # Generate synthetic policy events: ~40% of states get a funding reform
    states_in_panel = df["state"].unique()
    rng = np.random.default_rng(7)
    treated_states = rng.choice(states_in_panel, size=max(1, len(states_in_panel) // 2), replace=False)
    years_available = sorted(df["year"].unique())

    # Stagger events across the middle of the panel
    event_records = []
    mid = len(years_available) // 2
    for i, state in enumerate(treated_states):
        event_yr = years_available[mid - 1 + (i % 3)]   # spread across 3 years
        event_records.append({
            "state": state,
            "policy_type": "funding_reform",
            "event_year": str(event_yr),
            "event_name": f"{state} Funding Reform",
        })
    events = pd.DataFrame(event_records)
    print(f"Generated {len(events)} synthetic funding reform events:")
    print(events.to_string(index=False))


In [ ]:
# Convert panel to list-of-dicts format for event_study module
panel_rows = df.assign(year=df["year"].astype(str)).to_dict("records")
events_rows = events.to_dict("records")

enriched_rows = add_relative_time(panel_rows, events_rows, "funding_reform")
enriched = pd.DataFrame(enriched_rows)
enriched["years_since_funding_reform"] = pd.to_numeric(
    enriched["years_since_funding_reform"], errors="coerce"
)
enriched["outcome_composite"] = outcome_composite(enriched)

# Split treated (has event year) vs control
treated   = enriched[enriched["years_since_funding_reform"].notna()].copy()
control   = enriched[enriched["years_since_funding_reform"].isna()].copy()

print(f"Treated district-years: {len(treated):,}")
print(f"Control district-years: {len(control):,}")


## Event-study plot: outcomes in relative time

In [ ]:
window = (-4, 5)
es = treated[
    treated["years_since_funding_reform"].between(*window)
].copy()

agg = (
    es.groupby("years_since_funding_reform")["outcome_composite"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
agg["se"] = agg["std"] / np.sqrt(agg["count"])
agg["lo"] = agg["mean"] - 1.96 * agg["se"]
agg["hi"] = agg["mean"] + 1.96 * agg["se"]

# Normalise to the period just before the event (t = -1)
baseline = agg.loc[agg["years_since_funding_reform"] == -1, "mean"]
baseline = float(baseline.iloc[0]) if len(baseline) else float(agg["mean"].iloc[0])
agg["mean_norm"] = agg["mean"] - baseline
agg["lo_norm"]   = agg["lo"]  - baseline
agg["hi_norm"]   = agg["hi"]  - baseline

fig, ax = plt.subplots(figsize=(11, 5))
ax.axvline(0, color="#e76f51", linewidth=2, linestyle="--", alpha=0.8, label="Reform enacted (t=0)")
ax.axhline(0, color="black",   linewidth=1, linestyle="--", alpha=0.4)
ax.fill_between(
    agg["years_since_funding_reform"],
    agg["lo_norm"], agg["hi_norm"],
    alpha=0.2, color="#2a9d8f", label="95% CI"
)
ax.plot(agg["years_since_funding_reform"], agg["mean_norm"],
        "o-", color="#2a9d8f", linewidth=2.5, markersize=8, label="Mean outcome (relative to t=−1)")

ax.set_xticks(range(*window, 1))
ax.set_xticklabels([f"t{k:+d}" for k in range(*window, 1)])
ax.set_xlabel("Years Relative to Funding Reform Enactment")
ax.set_ylabel("Outcome Change (composite units, relative to t=−1)")
ax.set_title(
    "Event Study: Funding Reform and District Outcomes\n"
    "Relative-time plot for treated states",
    fontweight="bold"
)
ax.legend()
plt.tight_layout()
plt.show()


## Pre-trend test (parallel trends assumption)

In [ ]:
# Test: are pre-event coefficients jointly zero?
# Simple version: regress outcome on relative-time dummies for pre-period only.
pre = es[es["years_since_funding_reform"] < 0].copy()
pre["rel"] = pre["years_since_funding_reform"].astype(int)
rel_vals = sorted(pre["rel"].unique())

# OLS: outcome ~ relative_time (numeric, pre-period only)
# H0: slope == 0 (no pre-trend)
valid = pre.dropna(subset=["outcome_composite", "rel"])
slope, intercept = np.polyfit(valid["rel"], valid["outcome_composite"], 1)
residuals = valid["outcome_composite"] - (slope * valid["rel"] + intercept)
n = len(valid)
se_slope = np.sqrt(residuals.var() / ((valid["rel"] - valid["rel"].mean()) ** 2).sum())
t_stat = slope / se_slope

print("Pre-trend test (OLS on relative time, pre-period only):")
print(f"  Slope:       {slope:+.5f} outcome units per year")
print(f"  SE:          {se_slope:.5f}")
print(f"  t-statistic: {t_stat:.2f}")
print()
if abs(t_stat) < 2:
    print("Pre-trend NOT statistically significant — parallel trends assumption is plausible.")
else:
    print("WARNING: Pre-trend IS significant — treated districts may not be on a parallel path.")


## Comparing treated vs. control districts post-reform

In [ ]:
# Compute yearly averages for treated and control
def yearly_mean(group_df, label):
    return (
        group_df
        .groupby("year")["outcome_composite"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .assign(
            se=lambda d: d["std"] / np.sqrt(d["count"]),
            group=label
        )
    )

treated_yearly = yearly_mean(treated.assign(year=treated["year"].astype(int)), "Treated (reform states)")
control_yearly = yearly_mean(control.assign(year=pd.to_numeric(control["year"], errors="coerce")), "Control (no reform)")
combined = pd.concat([treated_yearly, control_yearly])

fig, ax = plt.subplots(figsize=(11, 5))
palette = {"Treated (reform states)": "#2a9d8f", "Control (no reform)": "#adb5bd"}
for group, color in palette.items():
    d = combined[combined["group"] == group].dropna(subset=["mean"]).sort_values("year")
    ax.plot(d["year"], d["mean"], "o-", color=color, linewidth=2.5, markersize=7, label=group)
    ax.fill_between(d["year"], d["mean"] - 1.96 * d["se"], d["mean"] + 1.96 * d["se"],
                    alpha=0.15, color=color)

# Mark reform years with vertical band
if len(events) > 0:
    ev_years = pd.to_numeric(events["event_year"], errors="coerce").dropna().unique()
    for yr in ev_years:
        ax.axvline(yr, color="#e76f51", linewidth=1, linestyle=":", alpha=0.5)

ax.set_xlabel("Year")
ax.set_ylabel("Mean Outcome Composite")
ax.set_title("Treated vs. Control District Outcomes Over Time\n(Dotted lines mark reform enactment years)", fontweight="bold")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()


## Takeaways

- **Pre-trends:** If the pre-reform relative-time coefficients are near zero and statistically non-significant, the parallel-trends assumption holds and event-study estimates are credible.
- **Post-reform effects:** An upward shift in outcomes after t=0, relative to the pre-period and to control states, is consistent with positive reform effects — but interpretation requires ruling out confounders.
- **Heterogeneity:** Effects likely vary by district poverty level, urbanicity, and implementation fidelity. Subgroup analysis is the natural next step.
- **Limitations:** This panel doesn't capture within-state rollout variation (some districts receive more funding than others). District-level treatment intensity (dollars received per pupil) would improve precision.

**Next step:** Run `eol-build-event-study` on a real panel to get demeaned outcome variables, then run a two-way fixed effects (TWFE) regression: `outcome_demeaned ~ post_reform × treated + year_FE`.
